In [ ]:
import numpy as np
import pandas as pd

import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from pathlib import Path

import matrices as M
import tuning as T
import waveforms as W
import os

from reexport import *
from replot import *
from resynthesize import *

In [ ]:
class Audio2WavetableFFT(AudioData):

    """
    1) __init__()
    2) helper functions
    3) main processes
    """

    def __init__(self, path, id, framesize = 2048, min_freq = 0):
        super().__init__(path, id, framesize, min_freq)
        self.audio_df = pd.DataFrame()
        self.harmonics = []
        self.frame = []
        self.period = 0

    # 2) helper functions ~~~~~~~~~~~~~~~~~~~~~~
    def _set_data_to_pow_2_srate(self, data):
        nearest_pow_2 = 2**np.ceil(np.log2(self.srate)).astype(int)

        if data.size < nearest_pow_2:
            data = np.pad(data,(0,nearest_pow_2 - data.size),mode = 'constant', constant_values=0)
        else:
            data = data[:nearest_pow_2]
        
        self.srate = nearest_pow_2
        df = pd.DataFrame(data = data, columns = ['signal'])
        return df
    
    def _set_zero_crossings(self, df):
        df = df.copy()
        df['zero_crossings'] = np.sign(df['signal']).diff().fillna(0)/2

        df['pos_x'] = df['zero_crossings']
        df.loc[df['pos_x'] <= 0, 'pos_x'] = np.nan
        df['pos_x'] = df['pos_x'] * df['signal']

        df['neg_x'] = df['zero_crossings']
        df.loc[df['neg_x'] >= 0, 'neg_x'] = np.nan
        df['neg_x'] = df['neg_x'].abs() * df['signal']

        df = df.drop('zero_crossings', axis = 1)
        
        return df
    
    def _shift_audio_signal(self, df, apply_window = True):
        df = df.copy()
        first_pos = 0
        _max = np.argmax(df['signal'])

        for i in range(_max,0, -1):
            if df['pos_x'].values[i] > 0 and i > first_pos:
                first_pos = i
                break

        last_pos = 0

        for i in range(len(df)-1, 0, -1):
            if df['neg_x'].values[i] < 0 and i > last_pos:
                last_pos = i
                break


        df['shift'] = df['signal']
        df.iloc[:first_pos,:].loc[:,'shift'] = 0

        df.iloc[last_pos:,:].loc[:,'shift'] = 0

        df['shift'] = df['shift'].shift(-first_pos).fillna(0)
        df = df.drop(['pos_x', 'neg_x'], axis = 1)

        if apply_window:
            w = np.hanning(last_pos-first_pos)
            w = np.pad(w, (0,len(df) - len(w)), mode='constant', constant_values= 0)
            df['shift'] = df['shift'] * w

        return df
    
    def _set_initial_peaks(self, audio_df):
        df = audio_df['spec'].copy()
        df = df.reset_index()
        s = df['spec']
        df['peak_cond'] = False
        df.loc[(s.shift(1) <= s) & (s > s.shift(-1)), 'peak_cond'] = True
        df['peaks'] = df['spec']
        df.loc[df['peak_cond'] == False, 'peaks'] = np.nan

        return df
    
    def _get_peak_stats_df(self, df):
        # #calculate the distance between one peak to every other peak in the spectrum ~~ begin
        dfa = df.loc[~df['peaks'].isna(),'index'].values
        dfb = df.loc[~df['peaks'].isna(),'peaks']
        X,Y = np.meshgrid(dfa,dfa)
        Z = X - Y
        #~~end

        dfb = dfb/dfb.max() #create a set of weights bound from [0,1] for every peak

        dfa = pd.DataFrame(Z, index = dfa, columns = dfa) * dfb #weight the distance calculations to determine what are true peaks

        dfa[dfa < 0] = 0 # some of the differences are 0s; only consider differences between peaks at a given index and following indices 
        dfa = dfa.transpose() # transpose for easier operations
        dfa = dfa.max(axis = 1) 
        dfa = dfa.reset_index() 
        dfa['certainty'] = 1 - np.abs(dfa['index'] - dfa[0])/dfa['index']

        return dfa
    
    def _get_period(self, df):
        # The period in samples 
        stats = df.describe()['certainty']
        mean = stats['mean']
        dev = stats['std']

        print(mean, dev)

        df[0] = True
        df.loc[df['certainty'] < mean + dev*1, 0] = False # turn off peaks below the threshold based on their distance for the max
        df = df[df[0] == True]
        df['diff'] = df['index'].diff().fillna(df['index']).copy()

        period = 0

        if len(df)>10:
            # This helps when the first harmonic isn't the highest
            # use statistics to determine avg peak to peak distance based on what peaks have 
            # the highest certainty score
            period = df['diff'].median().astype(int) 
        else:
            period = df['index'].values[np.argmax(df['certainty'])].astype(int)
        
        return period
    
    def _clean_spectrum(self, df):
        df['peaks'] = df['spec']
        df.loc[(df.index % self.period != 0) | (df.index == 0 ),'peaks'] = np.nan
        df['peaks'] = df['peaks'].fillna(0)
        ind = df['peaks'].ne(0).idxmax() - 1 # shift to first harmonic not 0 indx
        df['peaks'] = df['peaks'].shift(-ind).fillna(0).copy()

        return df

        
    

    # 3) main processes~~~~~~~~~~~~~~~~~~~~~~``
    def create_audio_data(self, apply_window = True):
        self.audio_df = pd.DataFrame()

        df = self._set_data_to_pow_2_srate(self.data)

        self.data = [] #no need to duplicate this in memory
        srate = self.srate

        df = self._set_zero_crossings(df)

        df = self._shift_audio_signal(df, apply_window = apply_window)      

        df['spec'] = np.abs(np.fft.fft(df['shift']))
        df.iloc[srate//2:srate,:].loc[:,'spec'] = 0
        
        self.audio_df = df

        return self.audio_df
    
    def resynthesize_data(self):
        output = []
        for i,amp in enumerate(self.harmonics):
            xs = np.linspace(0,2*np.pi,self.framesize, endpoint=False) * i
            output.append(amp * np.sin(xs))
        
        output = np.array(output)
        output = output.sum(axis=0)
        output = output/np.abs(output).max()
        self.frame = output

    def set_harmonics(self, audio_df, custom_period = 0):
        # get peaks from the spectrum
        df = self._set_initial_peaks(audio_df)

        df_stats = self._get_peak_stats_df(df)
        
        if custom_period <= 0:
            self.period = self._get_period(df_stats)
        else:
            self.period = custom_period

        #stats need to be reevaluated to better determine the first harmonic.

        df = self._clean_spectrum(df)

        self.harmonics = df['peaks'][df['peaks'] > 0].copy().reset_index(drop=True)
        self.audio_df['peaks'] = df['peaks']

        return self.audio_df
        
    
    def process(self):
        # try:
        self.create_audio_data()
        self.set_harmonics(self.audio_df)
        self.resynthesize_data()

        # except Exception as e:
        #     print(f'\n~~~Error at file {self.name}, id {self.id}~~~')
        #     print(e)
        #     print('~~~~~~\n')

        

    def export(self, path = "./", prepend = ""):
        if(len(self.frame) == self.framesize):
            export_wavetable(self.frame, path = path, filename = self.name, prepend = prepend)
        else:
            print(f"\nCould not export incompleted or partial wavetable of file {self.name}\n")
    
    def __str__(self):
        return f'\nfile name: {self.name}\
                \nsample rate: {self.srate}\
                \nanalysis period: {self.period}\
                \nharmonics : {len(self.harmonics)}\n'

In [ ]:
def audio_plot_fft(d):
    
    df = d.audio_df
    fig = make_subplots(
        rows=1,
        cols=4,
        subplot_titles=("Audio", f"Frequency Spectrum", "Harmonics","Wavetable")
    )

    # Line plot
    fig.add_trace(
        go.Scatter(
            x=np.arange(len(df)),
            y=df['shift'],
            mode="lines",
            marker=dict(color='rgb(51, 98, 163)')
        ),
        row=1,
        col=1
    )

    # Bar plot 1
    fig.add_trace(
        go.Scatter(
            x=np.arange(len(df['spec'])),
            y=df['spec'],
            mode="lines",
            marker=(dict(color='rgb(163, 60, 60)'))
        ),
        row=1,
        col=2
    )

    # Bar plot 2
    fig.add_trace(
        go.Bar(
            x=np.arange(len(d.harmonics))+ 1,
            y=d.harmonics,
            marker=(dict(color='rgb(31, 161, 91)'))
        ),
        row=1,
        col=3
    )

    fig.add_trace(
        go.Scatter(
            x=np.arange(len(d.frame)),
            y=d.frame,
            mode="lines",
            marker=(dict(color='rgb(101, 39, 156)'))
        ),
        row=1,
        col=4
    )

    # fig.update_xaxes(title_text=x_title)
    # fig.update_yaxes(title_text=y_title)

    fig.update_layout(
        title=d.name,
        height=400
    )

    return fig

In [ ]:
files = get_files("data")
data = []
for i, f in enumerate(files):
    a = Audio2WavetableFFT(f,i, min_freq = 0)
    a.process()
    if a.period > 0:
        data.append(a)
    else:
        print(f"File Skipped. Could not extract tonal information with the following data {a}")


In [ ]:
for i, f in enumerate(files):
    print(i,f)

In [ ]:
i = 5
# data[i].reprocess_ind(48)
f1 = audio_plot_fft(data[i])
print(data[i])

In [ ]:
f1.show()

In [ ]:
f = stack_plots(data)

In [ ]:
f.show()

In [ ]:
# for d in data:
#     d.export(path = "output", prepend="ct_")

consolidated_export(data, path = 'output', filename = 'guitar')

In [ ]:
for d in data:
    if list is type(d.hspectrum):
        print(d.id)

In [ ]:
data[49].fundamental_ind

In [ ]:
ex = np.roll(data[0].nspectrum,-123)
ex[-123:] = 0

ex

In [ ]:
a = Audio2WavetableFFT(files[28], 0)

In [ ]:
a.analyze_audio()
df = a.audio_df

fig = go.Figure()
fig.add_trace(go.Scatter(
        x=df.index, # Pass the coordinate as a list
        y=df['signal'], # Pass the coordinate as a list
        mode="lines")
)
fig.add_trace(go.Scatter(
        x=[np.argmax(df['signal'])], # Pass the coordinate as a list
        y=[df['signal'].max()], # Pass the coordinate as a list
        mode="markers")
)
fig.add_trace(go.Scatter(
        x=df.index, # Pass the coordinate as a list
        y=df['shift'], # Pass the coordinate as a list
        mode="lines")
)
fig.add_trace(go.Scatter(
        x=df.index, # Pass the coordinate as a list
        y=df['windowed'], # Pass the coordinate as a list
        mode="lines")
)
fig.add_trace(go.Scatter(
        x=df.index, # Pass the coordinate as a list
        y=df['spec'], # Pass the coordinate as a list
        mode="lines")
)

In [ ]:
# get peaks from the spectrum
df2 = df['spec'].copy()
df2 = df2.reset_index()
s = df2['spec']
df2['peak_cond'] = False
df2.loc[(s.shift(1) <= s) & (s > s.shift(-1)), 'peak_cond'] = True
df2['peaks'] = df2['spec']
df2.loc[df2['peak_cond'] == False, 'peaks'] = np.nan


# #calculate the distance between one peak to every other peak in the spectrum ~~ begin
dfa = df2.loc[~df2['peaks'].isna(),'index'].values
dfb = df2.loc[~df2['peaks'].isna(),'peaks']
X,Y = np.meshgrid(dfa,dfa)
Z = X - Y
#~~end

dfb = dfb/dfb.max() #create a set of weights bound from [0,1] for every peak

dfa = pd.DataFrame(Z, index = dfa, columns = dfa) * dfb #weight the distance calculations to determine what are true peaks

dfa[dfa < 0] = 0 # some of the differences are 0s; only consider differences between peaks at a given index and following indices 
dfa = dfa.transpose() # transpose for easier operations
dfa = dfa.max(axis = 1) 
dfa = dfa.reset_index() 
dfa['certainty'] = 1 - np.abs(dfa['index'] - dfa[0])/dfa['index']

stats = dfa.describe()['certainty']
mean = stats['mean']
dev = stats['std']

print(mean, dev)

dfa[0] = True
dfa.loc[dfa['certainty'] < mean + dev*1, 0] = False # turn off peaks below the threshold based on their distance for the max
dfa = dfa[dfa[0] == True]
dfa['diff'] = dfa['index'].diff().fillna(dfa['index'])
if len(dfa)>10:
    period = dfa['diff'].median().astype(int)
else:
    period = dfa['index'].values[np.argmax(dfa['certainty'])].astype(int)

print(period)

df2['peaks'] = df2['spec']
df2.loc[(df2.index % period != 0) | (df2.index == 0 ),'peaks'] = np.nan
df2['peaks'] = df2['peaks'].fillna(0)
ind = df2['peaks'].ne(0).idxmax() - 1 # shift to first harmonic not 0 indx
df2['peaks'] = df2['peaks'].shift(-ind).fillna(0)

dfh = df2['peaks'][df2['peaks'] > 0].copy().reset_index(drop=True)



In [ ]:
fig = go.Figure()
fig.add_trace(go.Scatter(
        x=df2.index, # Pass the coordinate as a list
        y=df2['spec'], # Pass the coordinate as a list
        mode="lines")
)
fig.add_trace(go.Scatter(
        x=df2.index, # Pass the coordinate as a list
        y=df2['peaks'], # Pass the coordinate as a list
        mode="lines")
)
# fig.add_trace(go.Scatter(
#         x=df2.index, # Pass the coordinate as a list
#         y=df2['threshold'], # Pass the coordinate as a list
#         mode="lines")
# )

In [ ]:
dfh

output = []
for i,amp in enumerate(dfh):
    xs = np.linspace(0,2*np.pi,2048, endpoint=False) * i
    output.append(amp * np.sin(xs))

output = np.array(output)
output = output.sum(axis=0)
output = output/np.abs(output).max()
frame = output

In [ ]:
px.line(frame)

In [ ]:
dfa